In [1]:
import torch

import numpy as np

edge_index = np.load(
    r"C:\Users\HP\Documents\python\bio informatics\final exam\ThreatGraphx\data\ml_ready\edge_index.npy"
)

print(edge_index.shape)

print(edge_index.max())

print(edge_index.min())

(2, 15573)
7501
0


### imports

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np

from torch_geometric.nn import SAGEConv

### Load Graph

In [3]:
edge_index = np.load(
    r"C:\Users\HP\Documents\python\bio informatics\final exam\ThreatGraphx\data\ml_ready\edge_index.npy"
)

edge_index = torch.tensor(
    edge_index,
    dtype=torch.long
)

print(edge_index.shape)

torch.Size([2, 15573])


### Graph Info

In [4]:
num_nodes = 7502

embedding_dim = 128

### Define GraphSAGE Model

In [5]:
class GraphSAGE(nn.Module):

    def __init__(self,
                 num_nodes,
                 embedding_dim):

        super().__init__()

        self.embedding = nn.Embedding(
            num_nodes,
            embedding_dim
        )

        self.conv1 = SAGEConv(
            embedding_dim,
            128
        )

        self.conv2 = SAGEConv(
            128,
            128
        )

    def forward(self, edge_index):

        x = self.embedding.weight

        x = self.conv1(
            x,
            edge_index
        )

        x = F.relu(x)

        x = self.conv2(
            x,
            edge_index
        )

        return x

### Create Model

In [6]:
model = GraphSAGE(
    num_nodes=7502,
    embedding_dim=128
)

model

GraphSAGE(
  (embedding): Embedding(7502, 128)
  (conv1): SAGEConv(128, 128, aggr=mean)
  (conv2): SAGEConv(128, 128, aggr=mean)
)

### Forward Pass

In [7]:
with torch.no_grad():

    embeddings = model(
        edge_index
    )

print(
    embeddings.shape
)

torch.Size([7502, 128])


-----------

### GRAPHSAGE TRAINING

### Create Positive Edges

In [8]:
positive_edges = edge_index.t()

print(
    positive_edges.shape
)

torch.Size([15573, 2])


### Create Negative Edges

In [9]:
num_edges = positive_edges.shape[0]

negative_edges = torch.randint(
    0,
    num_nodes,
    (num_edges, 2)
)

print(
    negative_edges.shape
)

torch.Size([15573, 2])


### Labels

In [10]:
positive_labels = torch.ones(
    num_edges
)

negative_labels = torch.zeros(
    num_edges
)

### Combine

In [11]:
all_edges = torch.cat(
    [
        positive_edges,
        negative_edges
    ],
    dim=0
)

all_labels = torch.cat(
    [
        positive_labels,
        negative_labels
    ],
    dim=0
)

print(all_edges.shape)

print(all_labels.shape)

torch.Size([31146, 2])
torch.Size([31146])


### Link Predictor

In [12]:
class LinkPredictor(nn.Module):

    def __init__(self,
                 embedding_dim):

        super().__init__()

        self.linear = nn.Linear(
            embedding_dim * 2,
            1
        )

    def forward(
        self,
        source,
        target
    ):

        x = torch.cat(
            [
                source,
                target
            ],
            dim=1
        )

        return self.linear(
            x
        )

### Create Predictor

In [13]:
predictor = LinkPredictor(
    128
)

### Optimizer

In [14]:
optimizer = torch.optim.Adam(

    list(model.parameters())
    +
    list(predictor.parameters()),

    lr=0.001
)

criterion = nn.BCEWithLogitsLoss()

### Training Loop

In [15]:
epochs = 20

for epoch in range(epochs):

    optimizer.zero_grad()

    embeddings = model(
        edge_index
    )

    src = embeddings[
        all_edges[:,0]
    ]

    dst = embeddings[
        all_edges[:,1]
    ]

    pred = predictor(
        src,
        dst
    ).squeeze()

    loss = criterion(
        pred,
        all_labels
    )

    loss.backward()

    optimizer.step()

    print(
        f"Epoch {epoch+1} "
        f"Loss: {loss.item():.4f}"
    )

Epoch 1 Loss: 0.7074
Epoch 2 Loss: 0.6525
Epoch 3 Loss: 0.6077
Epoch 4 Loss: 0.5683
Epoch 5 Loss: 0.5326
Epoch 6 Loss: 0.5005
Epoch 7 Loss: 0.4717
Epoch 8 Loss: 0.4461
Epoch 9 Loss: 0.4232
Epoch 10 Loss: 0.4024
Epoch 11 Loss: 0.3831
Epoch 12 Loss: 0.3656
Epoch 13 Loss: 0.3498
Epoch 14 Loss: 0.3357
Epoch 15 Loss: 0.3228
Epoch 16 Loss: 0.3111
Epoch 17 Loss: 0.3004
Epoch 18 Loss: 0.2908
Epoch 19 Loss: 0.2820
Epoch 20 Loss: 0.2739


### Evaluate GraphSAGE

In [16]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

### Prediction Model

In [17]:
model.eval()

predictor.eval()

LinkPredictor(
  (linear): Linear(in_features=256, out_features=1, bias=True)
)

### Generate Predictions

In [18]:
with torch.no_grad():

    embeddings = model(
        edge_index
    )

    src = embeddings[
        all_edges[:,0]
    ]

    dst = embeddings[
        all_edges[:,1]
    ]

    logits = predictor(
        src,
        dst
    ).squeeze()

    probs = torch.sigmoid(
        logits
    )

### Convert To Labels

In [19]:
pred_labels = (
    probs > 0.5
).int()

### Convert To Numpy

In [20]:
y_true = all_labels.numpy()

y_pred = pred_labels.numpy()

y_prob = probs.numpy()

### Metrics

In [21]:
accuracy = accuracy_score(
    y_true,
    y_pred
)

precision = precision_score(
    y_true,
    y_pred
)

recall = recall_score(
    y_true,
    y_pred
)

f1 = f1_score(
    y_true,
    y_pred
)

auc = roc_auc_score(
    y_true,
    y_prob
)

### Results

In [23]:
print("Accuracy :", accuracy)

print("Precision:", precision)

print("Recall   :", recall)

print("F1 Score :", f1)

print("AUC      :", auc)

Accuracy : 0.9019777820586913
Precision: 0.8498770400178851
Recall   : 0.9764335709240352
F1 Score : 0.9087703570894965
AUC      : 0.9513883237254204


### Save GraphSAGE

In [26]:
from pathlib import Path

project = Path(
    r"C:\Users\HP\Documents\python\bio informatics\final exam\ThreatGraphx"
)

model_dir = project / "models"

model_dir.mkdir(exist_ok=True)

torch.save(
    model.state_dict(),
    model_dir / "graphsage_model.pth"
)

torch.save(
    predictor.state_dict(),
    model_dir / "graphsage_predictor.pth"
)

print("GraphSAGE saved")

GraphSAGE saved


In [27]:
import os

print(
    os.listdir(model_dir)
)

['DistMult', 'graphsage_model.pth', 'graphsage_predictor.pth', 'TransE']
